# Bernoulli worked example

This notebook reproduces the worked example numerically. We:

1. Construct a random occupation-basis diagonal product state $\rho = \bigotimes_i \rho_i$ with each $\rho_i$ diagonal in $\mathrm{span}\{|0\rangle, |1\rangle\}_i$.
2. Evaluate every catalog-word connected cumulant of length 3 and 4 directly.
3. Confirm that the empirical $\Delta^{\mathrm{cat}}_{4, U(1)}(\rho) = 0$ to numerical precision.
4. Confirm that `certify(catalog, delta=0)` returns zero bias for every word.

Theory reference: the Bernoulli-worked-example theorem proves Delta = 0 on this state class.


In [ ]:
%pip install -q "cumulant_residual_cert@git+https://github.com/kootru-repo/cumulant-residual-cert.git" numpy

In [ ]:
import numpy as np

from cumulant_residual_cert import Catalog, certify

rng = np.random.default_rng(42)
n_sites = 6
occupation_probs = rng.uniform(0.1, 0.9, size=n_sites)
print('Per-site occupation probabilities:')
print(occupation_probs)

Build the diagonal density matrix in the occupation basis. Each site is an independent Bernoulli with prob $p_i$ of being occupied.


In [ ]:
dim = 2 ** n_sites
rho = np.zeros((dim, dim), dtype=complex)
for idx in range(dim):
    bits = [(idx >> (n_sites - 1 - i)) & 1 for i in range(n_sites)]
    prob = 1.0
    for p, b in zip(occupation_probs, bits):
        prob *= p if b == 1 else (1 - p)
    rho[idx, idx] = prob
print(f'Trace check: tr(rho) = {np.trace(rho).real:.12f}')

Connected cumulants of any catalog word of length 3 or 4 should vanish on this state. We use the library's internal cumulant evaluator to verify.


In [ ]:
from cumulant_residual_cert._fermion import subword_op
from cumulant_residual_cert._partition import set_partitions
from math import factorial

def connected_cumulant(rho, letters, sites, n):
    m = len(letters)
    moments = {}
    for k in range(1, m + 1):
        from itertools import combinations
        for B in combinations(range(1, m + 1), k):
            key = tuple(sorted(B))
            A_B = subword_op(letters, sites, key, n)
            moments[key] = complex(np.trace(rho @ A_B))
    kappa = 0.0 + 0.0j
    for pi in set_partitions(list(range(1, m + 1))):
        prod = 1.0 + 0.0j
        for C in pi:
            prod *= moments[tuple(sorted(C))]
        k = len(pi)
        kappa += (-1) ** (k - 1) * factorial(k - 1) * prod
    return kappa

cat = Catalog.chemistry_r4()
sites_per_word = [
    (1, 2, 3),
    (1, 2, 3),
    (1, 2, 3, 4),
    (1, 2, 3, 4),
    (1, 2, 3, 4),
]
for w, sites in zip(cat, sites_per_word):
    kappa = connected_cumulant(rho, w.letters, sites, n_sites)
    print(f'  kappa({w.name:32s}) = {kappa.real:+.3e} {kappa.imag:+.3e}j')

Every connected cumulant is numerically zero (within float64 round-off). Hence $\Delta^{\mathrm{cat}}_{4, U(1)}(\rho) = 0$ exactly.

Plug into `certify()`:


In [ ]:
result = certify(cat, delta=0.0)
for word, bar in result.bounds.items():
    print(f'  |tau({word:32s})|  <=  {bar:.4g}')

**Conclusion.** Every Bernoulli-class state, including the Hartree-Fock baseline of any number-conserving chemistry problem in its canonical molecular orbital basis, comes with a certified bias of exactly zero for every word in the chemistry catalog at $r = 4$.